In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

from datetime import datetime
import datetime as dt
import warnings
warnings.filterwarnings('ignore')

import scipy.stats as stats
import statsmodels.api as sm
from scipy.stats import norm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    StackingRegressor,
    ExtraTreesRegressor,
    BaggingRegressor,
    GradientBoostingRegressor
)
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import pickle

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df_energy = pd.read_csv("/content/drive/MyDrive/energydata_complete.csv")

In [ ]:
df_energy.head()

In [ ]:
df_energy.info()
df_energy.shape

In [ ]:
df_energy.describe()

In [ ]:
Columns = df_energy.columns
Columns

dup = df_energy.pivot_table(index=['date', 'Appliances', 'lights', 'T1', 'RH_1', 'T2', 'RH_2', 'T3','RH_3', 'T4', 'RH_4', 'T5', 'RH_5', 'T6', 'RH_6', 'T7', 'RH_7', 'T8',
                                'RH_8', 'T9', 'RH_9', 'T_out', 'Press_mm_hg', 'RH_out', 'Windspeed','Visibility', 'Tdewpoint', 'rv1', 'rv2'],aggfunc='size')

print(dup)

In [ ]:
dup_Count =  len(df_energy)-len(df_energy.drop_duplicates())

In [ ]:
dup_count1 = df_energy[df_energy.duplicated()].shape
dup_count1

In [ ]:
# Missing Values/Null Values Count
Null_values = df_energy.isnull().sum()
Null_values

In [ ]:
# Visualizing the missing values
plt.figure(figsize=(10,10))
sns.displot(
    data=df_energy.isna().melt(value_name="missing"),
    y="variable",
    hue="missing",
    multiple="fill",
    aspect=1.25
)
plt.savefig("visualizing_missing_data_with_barplot_Seaborn_distplot.png", dpi=100)



## Understanding your variables

In [ ]:
df_energy.columns

In [ ]:
df_energy.describe()

### Variable description
Date time year-month-day hour:minute:second

Appliances- energy use in Wh

Lights- energy use of light fixtures in the house in Wh

T1- Temperature in kitchen area, in Celsius

RH_1- Humidity in kitchen area, in %

T2-Temperature in living room area, in Celsius

RH_2- Humidity in living room area, in %

T3- Temperature in laundry room area

RH_3- Humidity in laundry room area, in %

T4- Temperature in office room, in Celsius

RH_4- Humidity in office room, in %

T5- Temperature in bathroom, in Celsius

RH_5- Humidity in bathroom, in %

T6- Temperature outside the building (north side), in Celsius

RH_6- Humidity outside the building (north side), in %

T7-Temperature in ironing room , in Celsius

RH_7- Humidity in ironing room, in %

T8- Temperature in teenager room 2, in Celsius

RH_8- Humidity in teenager room 2, in %

T9-Temperature in parents room, in Celsius

RH_9- Humidity in parents room, in %

To-Temperature outside (from Chievres weather station), in Celsius

Pressure (from Chievres weather station)- in mm Hg

RH_out- Humidity outside (from Chievres weather station), in %

Wind speed (from Chievres weather station)- in m/s

Visibility (from Chievres weather station)-in km

Tdewpoint (from Chievres weather station)- Â°C

rv1- Random variable 1, nondimensional

rv2- Random variable 2, nondimensional

Where indicated, hourly data (then interpolated) from the nearest airport weather station (Chievres Airport, Belgium) was downloaded from a public data set from Reliable Prognosis, rp5.ru. Permission was obtained from Reliable Prognosis for the distribution of the 4.5 months of weather data.

Check Unique Values for each variable.

In [ ]:
def get_unique_values(df_energy):
    unique_values=df_energy.apply(pd.Series.unique)
    return unique_values

unq_values=get_unique_values(df_energy)

for i in df_energy.columns.tolist():
    print("No of unique values in",i,"is",df_energy[i].nunique())

### 3. Data Wrangling
Data Wrangling Code

In [ ]:
#rename the columns
df_energy.rename(columns={'T1': 'temp_kitchen', 'RH_1':'hu_Kitchen', 'T2':'temp_living_room', 'RH_2': 'hu_living', 'T3':'temp_Laundry_room',
       'RH_3':'hu_laundry', 'T4':'temp_office_room', 'RH_4':'hu_office', 'T5':'temp_bathroom', 'RH_5':'hu_bath', 'T6':'temp_build_out'
       , 'RH_6':'hu_build_out', 'T7':'temp_ironing_room', 'RH_7':'hu_ironing_room', 'T8':'temp_teen_room',
       'RH_8':'hu_teen', 'T9':'temp_parents_room', 'RH_9':'hu_parent', 'T_out':'temp_out', 'RH_out':'out_humidity'},inplace = True)


In [ ]:
df_energy.columns

In [ ]:
df_energy.columns

In [ ]:
df_energy.head()

In [ ]:
df_energy.describe()

In [ ]:

import pandas as pd

# Assuming your dataframe has a 'timestamp' column
# To generate a sample dataframe for demonstration:


# Convert the 'timestamp' column to datetime
df_energy['date']=pd.to_datetime(df_energy['date'])

#Getting the months and days from date

df_energy['month'] = df_energy['date'].dt.month
df_energy['weekday'] = df_energy['date'].dt.weekday
df_energy['hour'] = df_energy['date'].dt.hour

#drop the date column
df_energy.drop('date',axis=1,inplace=True)


# Print the updated dataframe
df_energy.head()

In [ ]:
#separate column list for better analysis
temp_cols=['temp_kitchen','temp_living_room','temp_Laundry_room','temp_office_room','temp_bathroom','temp_build_out','temp_ironing_room','temp_teen_room','temp_parents_room']
hu_cols=['hu_Kitchen','hu_living','hu_laundry', 'hu_office','hu_bath','hu_build_out','hu_ironing_room','hu_teen','hu_parent']
light_cols=['light']
weather_cols=['temp_out','out_humidity',"Tdewpoint","Press_mm_hg","Windspeed","Visibility"]
date_col = ['month','weekday','hour']
random_col = ["rv1","rv2"]

## 4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables

### 1

In [ ]:
# Chart - 1 visualization code
#Dependent varaible "Appliance"
plt.figure(figsize=(7,7))
sns.distplot(df_energy['Appliances'], color = 'Blue')

### 2

In [ ]:
# Chart - 2 visualization code
# Dependent variable 'Price'
plt.figure(figsize=(5,5))
sns.distplot(np.log10(df_energy['Appliances']),color="y")

### 3

In [ ]:

# Chart - 3 visualization code
#@title Default title text
# plot a bar plot for each numerical feature count (except car_ID)

for col in temp_cols[:-1]:
    fig = plt.figure(figsize=(9,6))
    ax = fig.gca()
    feature = (df_energy[col])
    feature.hist(bins=100, ax = ax)
    ax.axvline(df_energy[col].mean(), color='magenta', linestyle='dashed', linewidth=2)
    ax.axvline(feature.median(), color='cyan', linestyle='dashed', linewidth=2)
    ax.set_title(col)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

n = len(temp_cols)
fig, ax = plt.subplots(n, 2, figsize=(20, n * 5))

for i, col in enumerate(temp_cols):
    # --- Left: Univariate Analysis ---
    sns.histplot(df_energy[col], kde=True, ax=ax[i, 0], color='skyblue')
    ax[i, 0].axvline(df_energy[col].mean(), color='orange', linestyle='--', label=f'Mean: {df_energy[col].mean():.2f}')
    ax[i, 0].axvline(df_energy[col].median(), color='green', linestyle='--', label=f'Median: {df_energy[col].median():.2f}')
    ax[i, 0].set_title(f'Distribution of {col}')
    ax[i, 0].legend()

    # --- Right: Bivariate Analysis (Scatter Plot) ---
    # Removed cbar=True to fix the AttributeError
    sns.scatterplot(data=df_energy, x=col, y='Appliances', color='green', ax=ax[i, 1])
    ax[i, 1].set_title(f'{col} vs Appliances Energy')

plt.tight_layout()
plt.show()

In [ ]:

for col  in temp_cols:
  skewness = df_energy[col].skew(axis=0, skipna = True)
  print("The skweness of " ,col ,"is",skewness  )

### 4

In [ ]:
# Chart - 4 visualization code
fig,ax=plt.subplots(len(hu_cols),2,figsize=(20,40))
for i,col in enumerate(hu_cols):
  #univariate of the columns
  dist=sns.distplot(df_energy[col],ax=ax[i,0])
  ax[i,0].axvline(df_energy[col].mean(), color='orange', linestyle='dashed', linewidth=2)
  ax[i,0].axvline(df_energy[col].median(), color='green', linestyle='dashed', linewidth=2)
  #Bivariate Analysis the Appliance Energy
  #lineplot
  scatter=sns.scatterplot(data=df_energy,x=col,y='Appliances',color='green',ax=ax[i,1]);


In [ ]:

for col  in hu_cols:
  skewness = df_energy[col].skew(axis=0, skipna = True)
  print("The skweness of " ,col ,"is",skewness  )

### 5

In [ ]:

# Chart - 5 visualization code
fig,ax=plt.subplots(len(weather_cols),2,figsize=(20,30))
for i,col in enumerate(weather_cols):
  #univariate of the columns
  dist=sns.distplot(df_energy[col],ax=ax[i,0])
  ax[i,0].axvline(df_energy[col].mean(), color='orange', linestyle='dashed', linewidth=2)
  ax[i,0].axvline(df_energy[col].median(), color='green', linestyle='dashed', linewidth=2)
  #Bivariate Analysis the Appliance Energy
  #lineplot
  scatter=sns.regplot(data=df_energy,x=col,y='Appliances',color='orange',ax=ax[i,1]);

In [ ]:
for col  in weather_cols:
  skewness = df_energy[col].skew(axis=0, skipna = True)
  print("The skweness of " ,col ,"is",skewness  )


### 6

In [ ]:
# Chart - 6 visualization code
fig,ax=plt.subplots(1,2,figsize=(15,5))

  #univariate of the columns
dist=sns.distplot(df_energy['lights'],ax=ax[0])
ax[0].axvline(df_energy[col].mean(), color='orange', linestyle='dashed', linewidth=2)
ax[0].axvline(df_energy[col].median(), color='green', linestyle='dashed', linewidth=2)
  #Bivariate Analysis the Appliance Energy
  #lineplot
scatter=sns.scatterplot(data=df_energy,x='lights',y='Appliances',color='orange',ax=ax[1]);

### 7

In [ ]:


# Chart - 7 visualization code
fig,ax=plt.subplots(len(random_col),2,figsize=(15,10))
for i,col in enumerate(random_col):
  #univariate of the columns
  dist=sns.distplot(df_energy[col],ax=ax[i,0])
  ax[i,0].axvline(df_energy[col].mean(), color='orange', linestyle='dashed', linewidth=2)
  ax[i,0].axvline(df_energy[col].median(), color='green', linestyle='dashed', linewidth=2)
  #Bivariate Analysis the Appliance Energy
  #lineplot
  scatter=sns.lineplot(data=df_energy,x=col,y='Appliances',color='orange',ax=ax[i,1]);


### Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code
plt.figure(figsize=(20,15))
correlation = df_energy.corr()
sns.heatmap(abs(correlation) ,annot =True , cmap = 'coolwarm', linewidth = 1)
plt.show()

In [ ]:

for col in temp_cols[:]:
  fig = plt.figure(figsize=(9,6))
  ax = fig.gca()
  feature = df_energy[col]
  label = df_energy['Appliances']
  correaltion = feature.corr(label)
  plt.scatter(x=feature,y =label)
  plt.xlabel(col)
  plt.ylabel('Appliances')
  ax.set_title('Appliances  vs ' + col + '- correlation: ' + str(correlation))
  z = np.polyfit(df_energy[col], df_energy['Appliances'], 1)
  y_hat = np.poly1d(z)(df_energy[col])

  plt.plot(df_energy[col], y_hat, "r--", lw=1)

plt.show()


In [ ]:

for col in temp_cols[1:-1]:
    fig = plt.figure(figsize=(9, 6))
    ax = fig.gca()
    feature = df_energy[col]
    label = df_energy['Appliances']
    correlation = feature.corr(label)
    plt.scatter(x=feature, y=label)
    plt.xlabel(col)
    plt.ylabel('Appliances')
    ax.set_title('Appliances vs ' + col + '- correlation: ' + str(correlation))
    z = np.polyfit(df_energy[col], df_energy['Appliances'], 1)
    y_hat = np.poly1d(z)(df_energy[col])

    plt.plot(df_energy[col], y_hat, "r--", lw=1)

plt.show()


In [ ]:
for col in temp_cols[:]:
  feature = df_energy[col]
  label = df_energy['Appliances']
  correaltion = feature.corr(label)
  print('the colreation of',col ,'with Appliances is',correaltion)


#### Corelation maps with humdity columns

In [ ]:
for col in hu_cols[:]:
  fig = plt.figure(figsize=(9,6))
  ax = fig.gca()
  feature = df_energy[col]
  label = df_energy['Appliances']
  correaltion = feature.corr(label)
  plt.scatter(x= feature,y = label)
  plt.xlabel(col)
  plt.ylabel('Appliances')
  ax.set_title('Appliances  vs ' + col + '- correlation: ' + str(correlation))
  z = np.polyfit(df_energy[col], df_energy['Appliances'], 1)
  y_hat = np.poly1d(z)(df_energy[col])

  plt.plot(df_energy[col], y_hat, "r--", lw=1)

plt.show()

In [ ]:

for col in hu_cols[:]:
  feature = df_energy[col]
  label = df_energy['Appliances']
  correaltion = feature.corr(label)
  print('the colreation of',col ,'with Appliances is',correaltion)

### Corealtion maps of weather columns

In [ ]:

for col in weather_cols[:]:
  fig = plt.figure(figsize=(9,6))
  ax = fig.gca()
  feature = df_energy[col]
  label = df_energy['Appliances']
  correaltion = feature.corr(label)
  plt.scatter(x= feature,y = label)
  plt.xlabel(col)
  plt.ylabel('Appliances')
  ax.set_title('Appliances  vs ' + col + '- correlation: ' + str(correlation))
  z = np.polyfit(df_energy[col], df_energy['Appliances'], 1)
  y_hat = np.poly1d(z)(df_energy[col])

  plt.plot(df_energy[col], y_hat, "r--", lw=1)

plt.show()


In [ ]:
for col in weather_cols[:]:
  feature = df_energy[col]
  label = df_energy['Appliances']
  correaltion = feature.corr(label)
  print('The corealtion of',col,'with the Appliances is ', correaltion)

#### Appliance Energy Column

In [ ]:
fig,ax=plt.subplots(2,2,figsize=(15,10))

#Distribution of Appliances
dist=sns.distplot(df_energy['Appliances'],ax=ax[0,0])
dist.set_title('Distribution of Appliances Energy')

#Average Appliances Energy over month
month_eng=pd.DataFrame(df_energy.groupby('month')['Appliances'].mean()).reset_index()
sns.violinplot(x=df_energy['month'],y=df_energy['Appliances'], ax=ax[0,1])

#Average Appliances Energy over weekdays
weekday_eng=pd.DataFrame(df_energy.groupby('weekday')['Appliances'].mean()).reset_index()
#sns.barplot(x=weekday_eng['weekday'],y=weekday_eng['Appliances'],ax=ax[1,0])
sns.boxplot(x=df_energy['weekday'],y=df_energy['Appliances'],ax=ax[1,0])
#Average Appliances Energy over hours
hour_eng=pd.DataFrame(df_energy.groupby('hour')['Appliances'].mean()).reset_index()
sns.barplot(x=hour_eng['hour'],y=hour_eng['Appliances'],ax=ax[1,1])

In [ ]:
## Chart - 15 - Pair Plot

# Pair Plot visualization code
plt.figure(figsize=(25,25))
pair=sns.pairplot(df_energy[temp_cols], diag_kind='kde')
plt.show()


In [ ]:
plt.figure(figsize=(25,25))
pair=sns.pairplot(df_energy[hu_cols], diag_kind='hist')
plt.show()

In [ ]:
plt.figure(figsize=(25,25))
pair=sns.pairplot(df_energy[weather_cols], diag_kind='hist')
plt.show()

### 5. Hypothesis Testing
Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.
There is no change in appliance energy consumption on weekdays and weekend.

There is no significant difference in the energy consumption for appliances between day and night.

The mean temperature in kitchen is greater than normal room temperature.

#### Hypothetical Statement - 1
1. State Your research hypothesis as a null hypothesis and alternate hypothesis.
Null :- There is no change in appliance energy consumption on weekdays and weekend

Alternate :- There is higher appliances energy consumption on weekends as compared to weekdays.

2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
data_weekday = df_energy[df_energy['weekday'] <= 5][['Appliances']]
data_weekend = df_energy[df_energy['weekday'] > 5][['Appliances']]

#Statistics Test and P-value
t_stat, p_val = stats.ttest_ind(data_weekday, data_weekend, equal_var=True)

print('T-Statistics value', t_stat)
print("P-Value", p_val)


In [ ]:
data_weekend

### Hypothetical Statement - 2
1. State Your research hypothesis as a null hypothesis and alternate hypothesis.
Null Hypothesis: The mean temperature in kitchen is greater than normal room temperature.

Alternate Hypothesis: The temperature in Kitchen is at max room temperature and it can not be above it.

2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
#collect the kitchen temperature
kitchen_rand= df_energy['temp_kitchen'].sample(1000)
N=len(kitchen_rand)
#mean of the sample
kitchen_rand_mean= kitchen_rand.mean()
#Normal room temperature
nrt = 20
# the standard deviation for population
std_pop = df_energy['temp_kitchen'].std()
# Perform Statistical Test to obtain P-Value

Z_stat = ((kitchen_rand_mean - nrt)/(std_pop/np.sqrt(N)))
print(f'Z_score is {Z_stat} ')

P_value=norm.cdf(Z_stat,0,1)
print(f'P_value is {P_value} ')

### Hypothetical Statement - 3
1. State Your research hypothesis as a null hypothesis and alternate hypothesis.
Null:- The average monthly appliances energy consumption is equal

Alternate:- There is a significant difference in the energy consumption for appliances between every month

2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
data_month_1 = df_energy[df_energy['month'] == 1][['Appliances']]
data_month_2 = df_energy[df_energy['month'] == 2][['Appliances']]
data_month_3 = df_energy[df_energy['month'] == 3][['Appliances']]
data_month_4 = df_energy[df_energy['month'] == 4][['Appliances']]
data_month_5 = df_energy[df_energy['month'] == 5][['Appliances']]


#we take a stastic state will compare the any of the 5 months data to all months data to know our hypotesis is true or false

##Statistics Test and P-value
t_stat, p_val = stats.ttest_ind(data_month_1, data_month_4,equal_var=True)

print('T-Statistics value', t_stat)
print("P-Value", p_val)

### 6. Feature Engineering & Data Pre-processing

#### Handling Missing Values & Missing Value Imputation


In [ ]:
Missing_Values = df_energy.isnull().sum()
Missing_Values

#### 2. Handling Outliers

In [ ]:
# Handling Outliers & Outlier treatments
df= df_energy.copy()
col_list = list(df.describe().columns)

#find the outliers using boxplot
plt.figure(figsize=(25, 20))
plt.suptitle("Box Plot", fontsize=18, y=0.95)

for n, ticker in enumerate(col_list):

    ax = plt.subplot(8, 4, n + 1)

    plt.subplots_adjust(hspace=0.5, wspace=0.2)

    sns.boxplot(x=df[ticker],color='pink', ax = ax)

    # chart formatting
    ax.set_title(ticker.upper())


In [ ]:
import pandas as pd
import numpy as np

def find_outliers_iqr(data):
    # Calculate the first quartile (Q1) and third quartile (Q3) for each column
    q1 = data.quantile(0.25)
    q3 = data.quantile(0.75)

    # Calculate the interquartile range (IQR) for each column
    iqr = q3 - q1

    # Calculate the lower and upper bounds for outliers for each column
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    # Check for outliers in each column and count the number of outliers
    outliers_count = (data < lower_bound) | (data > upper_bound)
    num_outliers = outliers_count.sum()

    return num_outliers


outliers_per_column = find_outliers_iqr(df_energy)
print("Number of outliers per column:")
print(outliers_per_column.sort_values(ascending = False))



In [ ]:



# Handling Outliers & Outlier treatments
for ftr in col_list:
  print(ftr,'\n')
  q_25= np.percentile(df[ftr], 25)
  q_75 = np.percentile(df[ftr], 75)
  iqr = q_75 - q_25
  print('Percentiles: 25th=%.3f, 75th=%.3f, IQR=%.3f' % (q_25, q_75, iqr))
  # calculate the outlier cutoff
  cut_off = iqr * 1.5
  lower = q_25 - cut_off
  upper = q_75 + cut_off
  print(f"\nlower = {lower} and upper = {upper} \n ")
  # identify outliers
  outliers = [x for x in df[ftr] if x < lower or x > upper]
  print('Identified outliers: %d' % len(outliers))
  #removing outliers
  if len(outliers)!=0:

    def bin(row):
      if row[ftr]> upper:
        return upper
      if row[ftr] < lower:
        return lower
      else:
        return row[ftr]



    df[ftr] =  df.apply (lambda row: bin(row), axis=1)
    print(f"{ftr} Outliers Removed")
  print("\n-------\n")

In [ ]:
plt.figure(figsize=(25, 20))
plt.suptitle("Box Plot without Outliers", fontsize=18, y=0.95)
#plot the all figures in loop with boxplot
for n, ticker in enumerate(col_list):

    ax = plt.subplot(8, 4, n + 1)

    plt.subplots_adjust(hspace=0.5, wspace=0.2)

    sns.boxplot(x=df[ticker],color='g' ,ax = ax)

    # chart formatting
    ax.set_title(ticker.upper())

### 4. Feature Manipulation & Selection
### 1. Feature Manipulation

In [ ]:
# Manipulate Features to minimize feature correlation and create new features
# create new features
# create a column average building temperature based on all temperature
df['Average_building_Temperature']=df[['temp_kitchen','temp_living_room','temp_Laundry_room','temp_office_room','temp_bathroom','temp_ironing_room','temp_teen_room','temp_parents_room']].mean(axis=1)
#create a column of difference between outside and inside temperature
df['Temperature_difference']=abs(df['Average_building_Temperature']-df['temp_build_out'])

#create a column average building humidity
df['Average_building_humidity']=df[['hu_Kitchen','hu_living','hu_laundry', 'hu_office','hu_bath','hu_ironing_room','hu_teen','hu_parent']].mean(axis=1)
#create a column of difference between outside and inside building humidity
df['Humidity_difference']=abs(df['hu_build_out']-df['Average_building_humidity'])


In [ ]:
# Select your features wisely to avoid overfitting
#drop rv1 and rv2
df.drop('rv1',axis=1,inplace=True)
df.drop('rv2',axis=1,inplace=True)
df.head()

In [ ]:
#create a function to check multicolinearity
from statsmodels.stats.outliers_influence import variance_inflation_factor
def calc_vif(X):
  vif = pd.DataFrame()
  vif['varabiles']= X.columns
  vif['VIF'] = [round(variance_inflation_factor(X.values,i),2) for i in range(X.shape[1])]
  return (vif)


In [ ]:
calc_vif(df[[i for i in df.describe().columns if i not in ['Appliances']]]).sort_values(by='VIF',ascending=False)

In [ ]:
calc_vif(df[[i for i in df.describe().columns if i not in ['Appliances','lights','temp_kitchen','temp_living_room','temp_Laundry_room','temp_office_room','temp_bathroom','hu_Kitchen','hu_living','hu_laundry','hu_office','hu_bath','temp_build_out','Average_building_Temperature']]]).sort_values(by='VIF',ascending=False)

In [ ]:
calc_vif(df[[i for i in df.describe().columns if i not in ['Appliances','lights','temp_kitchen','temp_living_room','temp_Laundry_room','temp_office_room','temp_bathroom','hu_Kitchen','hu_living','hu_laundry','hu_office','hu_bath','temp_build_out','Average_building_Temperature','Press_mm_hg','temp_parents_room','Average_building_humidity','temp_ironing_room','out_humidity']]]).sort_values(by='VIF',ascending=False)


In [ ]:
#check Multicolinearity
calc_vif(df[[i for i in df.describe().columns if i not in ['Appliances','lights','temp_kitchen','temp_living_room','temp_Laundry_room','temp_office_room','temp_bathroom','hu_Kitchen','hu_living','hu_laundry','hu_office','hu_bath','temp_build_out','Average_building_Temperature','Press_mm_hg','temp_parents_room','Average_building_humidity','temp_ironing_room','out_humidity','hu_teen','hu_parent','temp_teen_room','hu_ironing_room']]]).sort_values(by='VIF',ascending=False)


In [ ]:
#check Multicolinearity
calc_vif(df[[i for i in df.describe().columns if i not in ['Appliances','lights','temp_kitchen','temp_living_room','temp_Laundry_room','temp_office_room','temp_bathroom','hu_Kitchen','hu_living','hu_laundry','hu_office','hu_bath','temp_build_out','Average_building_Temperature','Press_mm_hg','temp_parents_room','Average_building_humidity','temp_ironing_room','out_humidity','hu_teen','hu_parent','temp_teen_room','hu_ironing_room','Temperature_difference','temp_out','Visibility']]]).sort_values(by='VIF',ascending=False)


In [ ]:
# Select your features wisely to avoid overfitting
df_removed=df[[i for i in df.describe().columns if i not in ['lights','temp_kitchen','temp_living_room','temp_Laundry_room','temp_office_room','temp_bathroom','hu_Kitchen','hu_living','hu_laundry','hu_office','hu_bath','temp_build_out','Average_building_Temperature','Press_mm_hg','temp_parents_room','Average_building_humidity','temp_ironing_room','out_humidity','hu_teen','hu_parent','temp_teen_room','hu_ironing_room','Temperature_difference','temp_out','Visibility']]]

df_removed.head()

### 5. Data Transformation

In [ ]:
# Transform Your data
#check distribution  of all independent features
for col in df_removed.describe().columns:
  fig=plt.figure(figsize=(9,6))
  ax=fig.gca()
  feature= (df_removed[col])
  sns.distplot(df_removed[col])
  ax.axvline(feature.mean(),color='magenta', linestyle='dashed', linewidth=2)
  ax.axvline(feature.median(),color='cyan', linestyle='dashed', linewidth=2)
  ax.set_title(col)
plt.show()

In [ ]:
columns  = ['Appliances','hu_build_out',	'Windspeed',	'Tdewpoint',	'month'	,'weekday'	,'hour'	,'Humidity_difference']
for col  in columns :
  skewness = df_removed[col].skew(axis=0, skipna = True)
  print("The skweness of " ,col ,"is",skewness  )

In [ ]:
# Transform Your data
df_removed['Appliances']=df_removed['Appliances'].apply(lambda x:np.log10(x+1))
df_removed['Windspeed']=df_removed['Windspeed'].apply(lambda x:np.log10(x+1))

In [ ]:

# check the distribution of the features after transformation
for col in df_removed.describe().columns:
  fig=plt.figure(figsize=(9,6))
  ax=fig.gca()
  feature= (df_removed[col])
  sns.distplot(df_removed[col])
  ax.axvline(feature.mean(),color='magenta', linestyle='dashed', linewidth=2)
  ax.axvline(feature.median(),color='cyan', linestyle='dashed', linewidth=2)
  ax.set_title(col)
plt.show()


### 6. Data Scaling

In [ ]:

from scipy.stats import zscore
# Scaling your data
Features  = ['hu_build_out',	'Windspeed',	'Tdewpoint',	'month'	,'weekday'	,'hour'	,'Humidity_difference']
X = df_removed[Features]
#X.shape

In [ ]:

y = df_removed['Appliances']
#y =np.log10(df_removed['Appliances'])

### 8. Data Splitting

In [ ]:

# Split your data to train and test. Choose Splitting ratio wisely.
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split( X,y , test_size = 0.2, random_state = 0)
print(X_train.shape)
print(X_test.shape)

In [ ]:
# Scaling your data
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled= scaler.transform(X_test)

## 7. ML Model Implementation
ML Model - 1

In [ ]:

# ML Model - 1 Implementation
from sklearn.linear_model import LinearRegression
# Fit the Algorithm
linear_reg =  LinearRegression().fit(X_train_scaled,y_train)

print(linear_reg.score(X_train_scaled,y_train))


# Predict on the model
y_pred = linear_reg.predict(X_test_scaled)

from sklearn.metrics import mean_squared_error

MSE  = mean_squared_error(10**(y_test), 10**(y_pred))
print("MSE :" , MSE)

RMSE = np.sqrt(MSE)
print("RMSE :" ,RMSE)

from sklearn.metrics import r2_score
r2 = r2_score(10**(y_test), 10**(y_pred))
print("R2 :" ,r2)
print("Adjusted R2 : ",1-(1-r2_score(10**(y_test), 10**(y_pred)))*((X_test.shape[0]-1)/(X_test.shape[0]-X_test.shape[1]-1)))

In [ ]:
# ML Model - 1 Implementation
#LinearRegresseion
lr=LinearRegression()
# Fit the LinearRegression
lr.fit(X_train_scaled,y_train)
#predict the target values of train data
lr_train=lr.predict(X_train_scaled)
# Predict the target values for the test data
y_pred_lr = lr.predict(X_test_scaled)
#Evaluate the model using
mse_lr_train=mean_squared_error(y_train,lr_train)
mse_lr_test = mean_squared_error(y_test, y_pred_lr)
r2_lr_train=r2_score(y_train,lr_train)
r2_lr_test = r2_score(y_test, y_pred_lr)

print("LinearRegression Mean Squared Error:",mse_lr_test)
print("LinearRegression R^2 Score:",r2_lr_test)


##Ridge
ridge = Ridge()
# Fit the Ridge model
ridge.fit(X_train_scaled, y_train)
#predict the target values of train data
r_train=ridge.predict(X_train_scaled)
# Predict  values for the test data
y_pred_r = ridge.predict(X_test_scaled)
# Evaluate the model using metrics
mse_r_train=mean_squared_error(y_train,r_train)
mse_r_test = mean_squared_error(y_test, y_pred_r)
r2_r_train=r2_score(y_train,r_train)
r2_r_test = r2_score(y_test, y_pred_r)

print("Ridge Mean Squared Error:",mse_r_test)
print("Ridge R^2 Score:",r2_r_test)


#Lasso
lasso = Lasso()
# Fit the Lasso model on the training data
lasso.fit(X_train_scaled, y_train)
#predict the target values of train data
l_train=ridge.predict(X_train_scaled)
# Predict the target values for the test data
y_pred_l = lasso.predict(X_test_scaled)
# Evaluate the model using metrics
mse_l_train=mean_squared_error(y_train,l_train)
mse_l_test = mean_squared_error(y_test, y_pred_l)
r2_l_train=r2_score(y_train,l_train)
r2_l_test = r2_score(y_test, y_pred_l)

print("Lasso Mean Squared Error:",mse_l_test)
print("Lasso R^2 Score:",r2_l_test)


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(10**(y_pred_lr))
plt.plot(10**(np.array((y_test))))
plt.legend(["Predicted","Actual"])
plt.show()

In [ ]:
# Visualizing evaluation Metric Score chart
#checking the coefficients
print("The Coefficients obtain from linearrgression model",lr.coef_)
print("The Coefficients obtain from Ridge model",ridge.coef_)
print("The Coefficients obtain from Lasso model",lasso.coef_)

#checking the intercepts
print("The Intercepts obtain from linearrgression model",lr.intercept_)
print("The Intercepts obtain from Ridge model",ridge.intercept_)
print("The Intercepts obtain from Lasso model",lasso.intercept_)

In [ ]:
# Visualizing evaluation Metric Score chart
## Plot the predicted vs actual values
plt.figure(figsize=(20,7))
plt.plot(((y_pred_lr)[500:550]),color='indigo')
plt.plot(((y_pred_r)[500:550]),color='green')
plt.plot(((y_pred_l)[500:550]),color='blue')
plt.plot((np.array((y_test)[500:550])),color='red')
plt.legend(["LinearRegression","Ridge","Lasso","Actual"])
plt.title("Sample of Actual vs Predicted of LinearRgression" )
plt.show()


2. Cross- Validation & Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)
ridge_params = {'alpha': [0.001, 0.01,0.1,0.5, 1,2,5]}
lasso_params = {'alpha': [0.0001, 0.001,0.01, 0.1,0.2,0.5,1]}

# Create Ridge and Lasso regression objects
ridge = Ridge()
lasso = Lasso()

# Create GridSearchCV objects
ridge_cv = GridSearchCV(ridge, param_grid=ridge_params, scoring='neg_mean_squared_error')
lasso_cv = GridSearchCV(lasso, param_grid=lasso_params, scoring='neg_mean_squared_error')

# Fit the models using GridSearchCV
ridge_cv.fit(X_train_scaled, y_train)
lasso_cv.fit(X_train_scaled, y_train)
# Get the best hyperparameters and fit the models again using the best hyperparameters
ridge_best = Ridge(alpha=ridge_cv.best_params_['alpha']).fit(X_train_scaled, y_train)
lasso_best = Lasso(alpha=lasso_cv.best_params_['alpha']).fit(X_train_scaled, y_train)

#predict the train data using the best models
y_train_ridge=ridge_best.predict(X_train_scaled)
y_train_lasso = lasso_best.predict(X_train_scaled)
# Predict  values for the test data using the best models
y_pred_ridge = ridge_best.predict(X_test_scaled)
y_pred_lasso = lasso_best.predict(X_test_scaled)

# Evaluate
mse_ridge_train = mean_squared_error(y_train, y_train_ridge)
mse_ridge_test = mean_squared_error(y_test, y_pred_ridge)
r2_ridge_train = r2_score(y_train, y_train_ridge)
r2_ridge_test = r2_score(y_test, y_pred_ridge)
mse_lasso_train = mean_squared_error(y_train, y_train_lasso)
mse_lasso_test = mean_squared_error(y_test, y_pred_lasso)
r2_lasso_train = r2_score(y_train, y_train_lasso)
r2_lasso_test = r2_score(y_test, y_pred_lasso)
# Print the evaluation metrics for both models
print("Ridge Regression - Best Alpha:" , ridge_cv.best_params_['alpha'])
print("Ridge Mean Squared Error:",(mse_ridge_test))
print("Ridge R^2 Score:",(r2_ridge_test))
mse_percent_ridge = mse_ridge_test * 100
r2_percent_ridge = r2_ridge_test * 100
print("Ridge - Mean squared error: {:.2f}%".format(mse_percent_ridge))
print("Ridge - R-squared: {:.2f}%".format(r2_percent_ridge))

print("\nLasso Regression - Best Alpha:",lasso_cv.best_params_['alpha'])
print("Lasso Mean Squared Error:",(mse_lasso_test))
print("Lasso R^2 Score:",(r2_lasso_test))

mse_percent_lasso = mse_lasso_test * 100
r2_percent_lasso = r2_lasso_test * 100
print("Lasso - Mean squared error: {:.2f}%".format(mse_percent_lasso))
print("Lasso - R-squared: {:.2f}%".format(r2_percent_lasso))
# Fit the Algorithm

# Predict on the model


In [ ]:
#create a list of metric score of linear regression
list_lr=['LinearRegression',mse_l_train,mse_l_test,r2_l_train,r2_l_test]
#create a empty dataframe for metric score columns
score = pd.DataFrame(columns = ['Model' , 'Train MSE', 'Test MSE' , 'Train R2_Score', 'Test R2_Score'])
#add the rows to the dataframe of linearrgression
score.loc[len(score)]=list_lr
#create a list of ridge metric score after hyperparameter tuning
list_ridge=['Ridge',mse_ridge_train,mse_ridge_test,r2_ridge_train,r2_ridge_test]
#add the rows to the dataframe
score.loc[len(score)]=list_ridge
#create a list of lasso metric score after hyperparameter tuning
list_lasso=['Lasso',mse_lasso_train,mse_lasso_test,r2_lasso_train,r2_lasso_test]
#add the rows to the dataframe
score.loc[len(score)]=list_lasso
print(score)


#### ML Model - 2

In [ ]:
from sklearn.linear_model import ElasticNet
#a * L1 + b * L2
#alpha = a + b and l1_ratio = a / (a + b)
elasticnet = ElasticNet(alpha=0.1, l1_ratio=0.5)

elasticnet.fit(X_train_scaled,y_train)

y_pred_en = elasticnet.predict(X_test)

MSE  = mean_squared_error(10**(y_test), 10**(y_pred_en))
print("MSE :" , MSE)

RMSE = np.sqrt(MSE)
print("RMSE :" ,RMSE)

r2 = r2_score(10**(y_test), 10**(y_pred_en))
print("R2 :" ,r2)
print("Adjusted R2 : ",1-(1-r2_score(10**(y_test), 10**(y_pred_en)))*((X_test.shape[0]-1)/(X_test.shape[0]-X_test.shape[1]-1)))

In [ ]:
# 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

# Visualizing evaluation Metric Score chart
plt.figure(figsize=(8,5))
plt.plot(10**(y_pred_en))
plt.plot((np.array(10**(y_test))))
plt.legend(["Predicted","Actual"])
plt.show()

In [ ]:
# 2. Cross- Validation & Hyperparameter Tuning

# ML Model - 1 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)
elastic = ElasticNet()
parameters = {'alpha': [1e-15,1e-13,1e-10,1e-8,1e-5,1e-4,1e-3,1e-2,1e-1,1,5,10,20,30,40,45,50,55,60,100],'l1_ratio':[0.3,0.4,0.5,0.6,0.7,0.8]}
elastic_regressor = GridSearchCV(elastic, parameters, scoring='neg_mean_squared_error',cv=5)
elastic_regressor.fit(X_train_scaled, y_train)

print("The best fit alpha value is found out to be :" ,elastic_regressor.best_params_)
print("\nUsing ",elastic_regressor.best_params_, " the negative mean squared error is: ", elastic_regressor.best_score_)
# Fit the Algorithm
y_pred_elastic = elastic_regressor.predict(X_test_scaled)
# Predict on the model
MSE  = mean_squared_error(10**(y_test), 10**(y_pred_elastic))
print("MSE :" , MSE)

RMSE = np.sqrt(MSE)
print("RMSE :" ,RMSE)

r2 = r2_score(10**(y_test), 10**(y_pred_elastic))
print("R2 :" ,r2)
print("Adjusted R2 : ",1-(1-r2_score(10**(y_test), 10**(y_pred_elastic)))*((X_test_scaled.shape[0]-1)/(X_test_scaled.shape[0]-X_test_scaled.shape[1]-1)))

y_train_elastic=elastic_regressor.predict(X_train_scaled)

mse_en_train = mean_squared_error(y_train, y_train_elastic)
mse_en_test = mean_squared_error(y_test, y_pred_elastic)
r2_en_train = r2_score(y_train, y_train_elastic)
r2_en_test = r2_score(y_test, y_pred_elastic)


In [ ]:
#create a list of decion tree regressor metric  score
dtr_list=['Elasticnet',mse_en_train,mse_en_test,r2_en_train,r2_en_test]
#add the rows by list
score.loc[len(score)]=dtr_list
score

#### ML Model-3

In [ ]:

# Visualizing evaluation Metric Score chart
# ML Model - 2 Implementation
# define the decision tree model
model = DecisionTreeRegressor(random_state=42)

#fit the model
model.fit(X_train_scaled, y_train)

#predictions on the test set
y_pred = model.predict(X_test_scaled)

#evaluate the model
mse_DT = mean_squared_error(y_test, y_pred)
r2_DT = r2_score(y_test, y_pred)

print("Mean squared error: ", mse_DT)
print("R-squared: ", r2_DT)

In [ ]:
# 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

# Visualizing evaluation Metric Score chart
sns.regplot(x=y_test, y=y_pred, color='green')

plt.xlabel('True Values')
plt.ylabel('Predictions')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(10**(y_pred))
plt.plot((np.array(10**(y_test))))
plt.legend(["Predicted","Actual"])
plt.show()

In [ ]:
# 2. Cross- Validation & Hyperparameter Tuning

 #ML Model - 2 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)
#hyperparameter
h_param= {'max_depth': [None, 15, 20, 25],
              'min_samples_split': [20, 25, 30],
              'min_samples_leaf': [4, 8, 12]}

#grid search cv
grid_search = GridSearchCV(model, h_param, cv=5, scoring='neg_mean_squared_error')
grid_search.fit(X_train_scaled, y_train)
print('Best parameters:', grid_search.best_params_)

dt_best = DecisionTreeRegressor(max_depth=grid_search.best_params_['max_depth'],
                                    min_samples_split=grid_search.best_params_['min_samples_split'],
                                    min_samples_leaf=grid_search.best_params_['min_samples_leaf'],
                                    random_state=0)

# Fit the Algorithm
dt_best.fit(X_train_scaled, y_train)

#predict on the training model
dtr_train=dt_best.predict(X_train_scaled)

# Predict on the model
y_pred = dt_best.predict(X_test_scaled)
#evaluate the model
mse_dsT_train=mean_squared_error(y_train,dtr_train)
mse_dsT_test = mean_squared_error(y_test, y_pred)
r2_dsT_train = r2_score(y_train, dtr_train)
r2_dsT_test = r2_score(y_test, y_pred)

print("Mean squared error: ", mse_dsT_test)
print("R-squared: ", r2_dsT_test)

In [ ]:
#create a list of decion tree regressor metric  score
dtr_list=['DecisionTree',mse_dsT_train,mse_dsT_test,r2_dsT_train,r2_dsT_test]
#add the rows by list
score.loc[len(score)]=dtr_list
score

#### ML Model - 4. Extra Tree Regression

In [ ]:
# Visualizing evaluation Metric Score chart
# ML Model - 4 Implementation
# define the decision tree model
model = ExtraTreesRegressor(random_state=42)

#fit the model
model.fit(X_train_scaled, y_train)

#predictions on the test set
y_pred = model.predict(X_test_scaled)

#evaluate the model
mse_br = mean_squared_error(y_test, y_pred)
r2_br = r2_score(y_test, y_pred)

print("Mean squared error: ", mse_br)
print("R-squared: ", r2_br)

In [ ]:
# 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

# Visualizing evaluation Metric Score chart
plt.scatter(y_test, y_pred)
plt.xlabel('Actual Target')
plt.ylabel('Predicted Target')
plt.title('Extra Tree Regressor - Actual vs Predicted')
sns.regplot(x=y_test, y=y_pred, scatter=False, color='red')
plt.show()

In [ ]:
# 2. Cross- Validation & Hyperparameter Tuning

 #ML Model - 4 Implementation with hyperparameter optimization techniques (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)
#hyperparameter
from sklearn.model_selection import GridSearchCV
#hyperparameter
h_param= { 'bootstrap': [True, False],
                          'max_depth': [70,100, None],
                          'criterion' :['squared_error'],
                          'max_features': ['log2', 'sqrt'],
                          'n_estimators': [10,1400,100]}

#grid search cv
grid_search = GridSearchCV(model, h_param, cv=5, scoring='neg_mean_squared_error')
grid_search.fit(X_train_scaled, y_train)
print('Best parameters:', grid_search.best_params_)

br_best = grid_search.best_estimator_

# Fit the Algorithm
br_best.fit(X_train_scaled, y_train)

#predict on the training model
br_train=dt_best.predict(X_train_scaled)

# Predict on the model
y_pred = br_best.predict(X_test_scaled)
#evaluate the model
mse_br_train=mean_squared_error(y_train,br_train)
mse_br_test = mean_squared_error(y_test, y_pred)
r2_br_train = r2_score(y_train, br_train)
r2_br_test = r2_score(y_test, y_pred)

print("Mean squared error: ", mse_br_test)
print("R-squared: ", r2_br_test)

In [ ]:
# 2. Cross-Validation & Hyperparameter Tuning
# ML Model - 4 Implementation with hyperparameter optimization techniques (GridSearchCV)

from sklearn.model_selection import GridSearchCV

# Hyperparameters
h_param = {
    'bootstrap': [True, False],
    'max_depth': [70, 100, None],
    'criterion': ['squared_error'],
    'max_features': ['log2', 'sqrt'],
    'n_estimators': [10, 100, 1400]  # ✅ Reordered for clarity (minor)
}

# Grid Search CV
grid_search = GridSearchCV(model, h_param, cv=5, scoring='neg_mean_squared_error')
grid_search.fit(X_train_scaled, y_train)

print('Best parameters:', grid_search.best_params_)

br_best = grid_search.best_estimator_

# Fit the best estimator
br_best.fit(X_train_scaled, y_train)

# Predict on training data
br_train = br_best.predict(X_train_scaled)  # ✅ Fixed: was dt_best.predict() — wrong variable name

# Predict on test data
y_pred = br_best.predict(X_test_scaled)

# Evaluate the model
mse_br_train = mean_squared_error(y_train, br_train)
mse_br_test  = mean_squared_error(y_test, y_pred)
r2_br_train  = r2_score(y_train, br_train)
r2_br_test   = r2_score(y_test, y_pred)

print("Train MSE:      ", mse_br_train)   # ✅ Added — was computed but never printed
print("Test MSE:       ", mse_br_test)
print("Train R-squared:", r2_br_train)    # ✅ Added — was computed but never printed
print("Test R-squared: ", r2_br_test)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV # Switched for speed

h_param = {
    'bootstrap': [True, False],
    'max_depth': [10, 50, 100, None],
    'max_features': ['log2', 'sqrt'],
    'n_estimators': [50, 100, 200] # Lowered for the search; you can increase later
}

# n_jobs=-1 will make this MUCH faster
random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=h_param,
    n_iter=20, # Only tries 20 random combinations
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1 # This will show you progress
)

random_search.fit(X_train_scaled, y_train)

print('Best parameters:', random_search.best_params_)
br_best = random_search.best_estimator_

# Note: No need to call br_best.fit() again, it's already fitted!
br_train = br_best.predict(X_train_scaled) # Fixed the 'dt_best' typo
y_pred = br_best.predict(X_test_scaled)

In [ ]:
#create a list of extra tree regressor metric  score
br_list=['ExtraTreeRegressor',mse_br_train,mse_br_test,r2_br_train,r2_br_test]
#add the rows by list
score.loc[len(score)]=br_list
score

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(15,10))
#plot the mse of all model
score.plot(x="Model", y=['Train MSE' , 'Test MSE'], kind="bar" , title = 'MSE Score Results',ax=ax[0])
#plot the r2_score of all model
score.plot(x="Model", y=['Train R2_Score' , 'Test R2_Score'], kind="bar" , title = 'R2 Score Results',ax=ax[1])